# 04b. Model Training


## 1. Data Loading & Splitting
Load prepared data and perform temporal train/validation/test splits.

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.metrics import precision_recall_curve, precision_score, recall_score, f1_score, roc_auc_score, average_precision_score
import joblib
import os

agg_model = pd.read_parquet('data/processed/modeling_ready.parquet')
prep_meta = joblib.load('models/prep_metadata.joblib')
surge_threshold = prep_meta['surge_threshold']
print(f'Loaded prepared data and surge_threshold ({surge_threshold:.2f})')


Loaded prepared data and surge_threshold (11.00)


In [2]:
available_years = sorted(agg_model["year"].unique())
print("Available years in modeling table:", available_years)

if 2025 in available_years and 2026 in available_years:
    train_mask = agg_model["year"] <= 2024
    val_mask = agg_model["year"] == 2025
    test_mask = agg_model["year"] >= 2026
elif 2025 in available_years:
    train_mask = agg_model["year"] <= 2024
    val_mask = agg_model["year"] == 2025
    test_mask = agg_model["year"] == 2025
else:
    cutoff = np.quantile(agg_model["time_bin"].view("int64"), 0.8)
    train_mask = agg_model["time_bin"].view("int64") < cutoff
    val_mask = agg_model["time_bin"].view("int64") >= cutoff
    test_mask = val_mask

feature_cols = [
    "month",
    "hour",
    "day_of_week",
    "is_weekend",
    "lag_calls_1",
    "lag_calls_2",
    "lag_calls_4",
    "lag_calls_8",
    "lag_calls_12",
    "lag_calls_24",
    "lag_cancel_rate_1",
    "lag_cancel_rate_4",
    "lag_cancel_rate_8",
    "lag_priority1_share_1",
    "lag_priority1_share_4",
    "lag_priority1_share_8",
    "rolling_calls_mean_4",
    "rolling_calls_mean_12",
    "rolling_calls_std_12",
    "rolling_cancel_mean_12",
    "rolling_priority1_mean_12",
    "unique_call_types",
]
target_col = "is_surge"

X_train = agg_model.loc[train_mask, feature_cols].copy()
y_train = agg_model.loc[train_mask, target_col].copy()
X_val = agg_model.loc[val_mask, feature_cols].copy()
y_val = agg_model.loc[val_mask, target_col].copy()
X_test = agg_model.loc[test_mask, feature_cols].copy()
y_test = agg_model.loc[test_mask, target_col].copy()

if X_val.empty:
    raise ValueError("Validation split is empty. Check year coverage in the dataset.")
if X_test.empty:
    raise ValueError("Test split is empty. Check year coverage in the dataset.")

print("Train rows:", len(X_train), " | surge rate:", round(y_train.mean(), 3))
print("Val rows:", len(X_val), " | surge rate:", round(y_val.mean(), 3))
print("Test rows:", len(X_test), " | surge rate:", round(y_test.mean(), 3))

Available years in modeling table: [np.int32(2022), np.int32(2023), np.int32(2024), np.int32(2025), np.int32(2026)]
Train rows: 103634  | surge rate: 0.277
Val rows: 34660  | surge rate: 0.215
Test rows: 11086  | surge rate: 0.231


## 2. Model Definition
Configure the preprocessing pipeline and model architectures.

In [3]:
categorical_cols = ["day_of_week"]
numeric_cols = [c for c in feature_cols if c not in categorical_cols]

preprocessor = ColumnTransformer(
    transformers=[
        ("num", StandardScaler(), numeric_cols),
        ("cat", OneHotEncoder(handle_unknown="ignore"), categorical_cols),
    ],
    remainder="drop",
)

baseline_model = Pipeline(
    steps=[
        ("preprocess", preprocessor),
        ("model", LogisticRegression(max_iter=2000, class_weight="balanced")),
    ]
)

main_model = Pipeline(
    steps=[
        ("preprocess", preprocessor),
        (
            "model",
            RandomForestClassifier(
                n_estimators=400,
                max_depth=10,
                min_samples_leaf=8,
                random_state=42,
                n_jobs=-1,
                class_weight="balanced_subsample",
            ),
        ),
    ]
)

## 3. Helper Functions
Define utility functions for threshold optimization and metric collection.

In [4]:
def best_f1_threshold(y_true: pd.Series, y_prob: np.ndarray) -> float:
    precision, recall, thresholds = precision_recall_curve(y_true, y_prob)
    f1_vals = (2 * precision * recall) / (precision + recall + 1e-12)
    if len(thresholds) == 0:
        return 0.5
    best_idx = int(np.nanargmax(f1_vals[:-1]))
    return float(thresholds[best_idx])


def collect_metrics(y_true: pd.Series, y_prob: np.ndarray, threshold: float) -> dict:
    y_pred = (y_prob >= threshold).astype(int)
    return {
        "precision": precision_score(y_true, y_pred, zero_division=0),
        "recall": recall_score(y_true, y_pred, zero_division=0),
        "f1": f1_score(y_true, y_pred, zero_division=0),
        "roc_auc": roc_auc_score(y_true, y_prob),
        "pr_auc": average_precision_score(y_true, y_prob),
    }

## 4. Model Training
Train the baseline and random forest models and optimize classification thresholds.

In [5]:
baseline_model.fit(X_train, y_train)
main_model.fit(X_train, y_train)

baseline_val_prob = baseline_model.predict_proba(X_val)[:, 1]
main_val_prob = main_model.predict_proba(X_val)[:, 1]

baseline_threshold = best_f1_threshold(y_val, baseline_val_prob)
main_threshold = best_f1_threshold(y_val, main_val_prob)

baseline_test_prob = baseline_model.predict_proba(X_test)[:, 1]
main_test_prob = main_model.predict_proba(X_test)[:, 1]

summary = pd.DataFrame(
    [
        {
            "model": "Logistic baseline",
            "split": "validation",
            **collect_metrics(y_val, baseline_val_prob, baseline_threshold),
        },
        {
            "model": "Random forest main",
            "split": "validation",
            **collect_metrics(y_val, main_val_prob, main_threshold),
        },
        {
            "model": "Logistic baseline",
            "split": "test",
            **collect_metrics(y_test, baseline_test_prob, baseline_threshold),
        },
        {
            "model": "Random forest main",
            "split": "test",
            **collect_metrics(y_test, main_test_prob, main_threshold),
        },
    ]
)

print(f"Baseline threshold (val F1-opt): {baseline_threshold:.3f}")
print(f"Main threshold (val F1-opt): {main_threshold:.3f}")
display(summary.round(3))

Baseline threshold (val F1-opt): 0.682
Main threshold (val F1-opt): 0.691


,model,split,precision,recall,f1,roc_auc,pr_auc
0,Logistic baseline,validation,0.789,0.801,0.795,0.962,0.894
1,Random forest main,validation,0.787,0.803,0.795,0.960,0.879
2,Logistic baseline,test,0.824,0.756,0.789,0.954,0.890
3,Random forest main,test,0.821,0.734,0.775,0.955,0.875


## 5. Model Export
Persist the trained models to disk.

In [6]:
import joblib
os.makedirs('models', exist_ok=True)
joblib.dump(main_model, 'models/random_forest_surge.joblib')
joblib.dump(baseline_model, 'models/logistic_baseline.joblib')

metadata = {
    'feature_cols': feature_cols,
    'numeric_cols': numeric_cols,
    'categorical_cols': categorical_cols,
    'main_threshold': main_threshold,
    'baseline_threshold': baseline_threshold,
    'surge_threshold': surge_threshold
}
joblib.dump(metadata, 'models/model_metadata.joblib')
print('Models and metadata saved.')


Models and metadata saved.
